<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Sinusoidal-Positional-Embedding/sinusoidal-q7-Question.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("sinusoidal-positional-embedding", ...)`


# Implement Sinusoidal Embeddings
###  Problem Statement
Transformers are order-agnostic — they see tokens like goldfish: no sense of sequence. To inject **position awareness** into the model, we use **Sinusoidal Positional Embeddings**, where each position in the sequence gets a unique deterministic vector. These vectors are computed using sine and cosine waves at different frequencies.

Your task is to implement the sinusoidal position encoding from "Attention Is All You Need" using PyTorch. Do not use a positional-encoding module from Fairseq or Hugging Face.

---

###  Requirements

1. **Define the Sinusoidal Embedding Class**
   - Implement a `SinusoidalPositionalEmbedding` class inheriting from `nn.Module`.
   - Initialize with `max_seq_len` and `d_model`, in that order.
   - Create a tensor `pe` of shape `(max_seq_len, d_model)`. Positions start at 0. `d_model` is even. For each frequency index `i = 0, 1, ..., d_model/2 - 1`, set `ω_i = 1 / 10000**(2i/d_model)`:
     - dimension `2i` gets `sin(pos * ω_i)`
     - dimension `2i+1` gets `cos(pos * ω_i)`
   - That same `ω_i` is shared by the pair. `i` is not the raw dimension index.

2. **Register as Buffer**
   - Use `self.register_buffer("pe", pe)` to store `pe` without treating it as a trainable parameter.

3. **Generate Encodings**
   - `forward(x)` receives `x` of shape `(batch_size, seq_len, d_model)`.
   - Use only `seq_len` (`x.shape[1]`). Ignore the values in `x`.
   - Return the first `seq_len` rows of `pe` with a leading size-1 batch dimension, shape `(1, seq_len, d_model)`, so the result broadcasts across the batch.

4. **Test the Embeddings**
   - Construct `SinusoidalPositionalEmbedding(max_seq_len, d_model)` with `max_seq_len = 100` and `d_model = 64`.
   - Pass an input of shape `(1, 50, 64)` and check that the output shape is `(1, 50, 64)`.

---

### Constraints

- ✅ Do not use Hugging Face, Fairseq, or any prebuilt positional-encoding module.
- ✅ Ensure the `pe` tensor is not a trainable parameter.
- ✅ Support any sequence length up to `max_seq_len`.
- ❌ Do not add these embeddings to token embeddings yet — this module only returns the positional table.

---

<details>
  <summary>💡 Hint</summary>

  - Use `torch.arange(0, max_seq_len).unsqueeze(1)` to create position indices.
  - Compute the shared frequencies with `torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))`. `arange(0, d_model, 2)` is `2i`, so each value is the `ω_i` for one sin/cos pair.
  - Write those sines into `pe[:, 0::2]` and the cosines into `pe[:, 1::2]`.
  - In `forward`, slice `self.pe[:seq_len]` and call `.unsqueeze(0)` so the batch dimension is 1.

</details>

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, max_seq_len: int, d_model: int):
        """
        Initializes the sinusoidal positional embedding.
        
        Args:
            max_seq_len (int): Maximum sequence length.
            d_model (int): Embedding dimension.
        """
        ...

    def forward(self, x):
        """
        Returns the positional embedding for a given input tensor.
        
        Args:
            x (Tensor): Input of shape (batch_size, seq_len, d_model).
                Only seq_len is used; the values in x are ignored.
        
        Returns:
            Tensor: Positional embeddings of shape (1, seq_len, d_model).
        """
        ...

In [ ]:
max_seq_len = 100
d_model = 64
seq_len = 50

custom_pos_emb = SinusoidalPositionalEmbedding(max_seq_len, d_model)

x = torch.zeros(1, seq_len, d_model)
positional_encoding = custom_pos_emb(x)

print(positional_encoding.shape)  # (1, 50, 64)
assert positional_encoding.shape == (1, 50, 64)